<a href="https://colab.research.google.com/github/Atul57/Data-Science-project/blob/main/Dynamic%20Graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

data = {
    "Company": ["ABC Corp"] * 15,

    "Department": [
        "IT","IT","IT",
        "HR","HR","HR",
        "Marketing","Marketing","Marketing",
        "Finance","Finance","Finance",
        "Operations","Operations","Operations"
    ],

    "Category": [
        "Salaries","Infrastructure","Training",
        "Salaries","Recruitment","Training",
        "Advertising","Events","Salaries",
        "Salaries","Software","Compliance",
        "Logistics","Maintenance","Salaries"
    ],

    "Amount":[
        500000,250000,50000,
        200000,80000,40000,
        300000,100000,150000,
        250000,120000,60000,
        400000,180000,350000
    ]
}

df = pd.DataFrame(data)
df

,Company,Department,Category,Amount
0,ABC Corp,IT,Salaries,500000
1,ABC Corp,IT,Infrastructure,250000
2,ABC Corp,IT,Training,50000
3,ABC Corp,HR,Salaries,200000
4,ABC Corp,HR,Recruitment,80000
5,ABC Corp,HR,Training,40000
6,ABC Corp,Marketing,Advertising,300000
7,ABC Corp,Marketing,Events,100000
8,ABC Corp,Marketing,Salaries,150000
9,ABC Corp,Finance,Salaries,250000


In [ ]:
import plotly.express as px

# Generate the Sunburst Chart
fig_sunburst = px.sunburst(
    df,
    path=['Company', 'Department', 'Category'],
    values='Amount',
    title="Proportional Breakdown: ABC Corp Spending",
    color='Department'
)

# Enlarge the chart slightly for better interaction
fig_sunburst.update_layout(width=800, height=600)
fig_sunburst.show()

In [ ]:
import plotly.graph_objects as go

# 1. Transform the flat dataframe into Source -> Target flows
tier1 = df.groupby(['Company', 'Department'])['Amount'].sum().reset_index()
tier1.columns = ['source', 'target', 'value']

tier2 = df.groupby(['Department', 'Category'])['Amount'].sum().reset_index()
tier2.columns = ['source', 'target', 'value']

# Combine flows
df_links = pd.concat([tier1, tier2], ignore_index=True)

# 2. Map the text names to unique integer IDs (required by Plotly Sankey)
unique_nodes = list(pd.unique(df_links[['source', 'target']].values.ravel('K')))
node_mapping = {node: i for i, node in enumerate(unique_nodes)}

df_links['source_id'] = df_links['source'].map(node_mapping)
df_links['target_id'] = df_links['target'].map(node_mapping)

# 3. Generate the Sankey Diagram
fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20, thickness=30,
        line=dict(color="black", width=0.5),
        label=unique_nodes,
        color="rgba(100, 150, 250, 0.8)"
    ),
    link=dict(
        source=df_links['source_id'],
        target=df_links['target_id'],
        value=df_links['value'],
        color="rgba(200, 200, 200, 0.4)"
    )
)])

fig_sankey.update_layout(
    title_text="Resource Flow: ABC Corp Budget Allocation",
    font_size=12, width=900, height=600
)
fig_sankey.show()